# Brain Tumor Detection with CNN

This notebook trains a convolutional neural network to classify brain MRI images as tumor / no tumor, with visualizations at each stage: sample data, training curves, a confusion matrix, and labeled predictions.

Run cells top to bottom with **Shift+Enter**.

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

DATASET_DIR = "./Dataset"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")
PREDICTION_DIR = os.path.join(DATASET_DIR, "prediction")
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15

## 2. Load data

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="binary"
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode="binary", shuffle=False
)

class_indices = train_generator.class_indices
index_to_label = {v: k for k, v in class_indices.items()}
print("Class mapping:", class_indices)

### Preview a batch of training images
Sanity-check that the images and labels line up before training.

In [ ]:
images, labels = next(train_generator)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, label in zip(axes.flatten(), images, labels):
    ax.imshow(img)
    ax.set_title(index_to_label[int(label)])
    ax.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()

train_generator.reset()

## 3. Build the model

In [ ]:
cnn = Sequential()
cnn.add(Conv2D(filters=32, kernel_size=3, activation="relu", input_shape=[224, 224, 3]))
cnn.add(MaxPooling2D(pool_size=2, strides=2))
cnn.add(Conv2D(filters=64, kernel_size=3, activation="relu"))
cnn.add(MaxPooling2D(pool_size=2, strides=2))
cnn.add(Conv2D(filters=128, kernel_size=3, activation="relu"))
cnn.add(MaxPooling2D(pool_size=2, strides=2))
cnn.add(Dropout(0.25))
cnn.add(Flatten())
cnn.add(Dense(units=128, activation="relu"))
cnn.add(Dropout(0.5))
cnn.add(Dense(units=1, activation="sigmoid"))

cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
cnn.summary()

## 4. Train

In [ ]:
history = cnn.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
)

cnn.save("brain_tumor_cnn.h5")

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Evaluate on the test set

In [ ]:
test_generator.reset()
predictions = cnn.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int).flatten()
true_classes = test_generator.classes
labels_in_order = [index_to_label[i] for i in range(len(index_to_label))]

print(classification_report(true_classes, predicted_classes, target_names=labels_in_order))

cm = confusion_matrix(true_classes, predicted_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_in_order)
disp.plot(cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()

## 6. Visualize predictions on the `prediction` folder
Shows each image with the model's prediction. Title is green if it matches the filename's expected label (files named like `yes1.jpg` / `no1.jpg`), red if it doesn't.

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

pred_files = sorted(
    f for f in os.listdir(PREDICTION_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
)

n = len(pred_files)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]

for ax, fname in zip(axes, pred_files):
    img_path = os.path.join(PREDICTION_DIR, fname)
    img = keras_image.load_img(img_path, target_size=IMAGE_SIZE)
    arr = keras_image.img_to_array(img) / 255.0
    arr_batch = np.expand_dims(arr, axis=0)

    result = cnn.predict(arr_batch, verbose=0)[0][0]
    predicted_label = index_to_label[int(result >= 0.5)]

    expected_label = "yes" if fname.lower().startswith("yes") else (
        "no" if fname.lower().startswith("no") else None
    )
    color = "black"
    if expected_label:
        color = "green" if predicted_label == expected_label else "red"

    ax.imshow(img)
    ax.set_title(f"{fname}\npredicted: {predicted_label} ({result:.2f})", color=color, fontsize=10)
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()